In [ ]:
# ACCOUNT A — AUGMENTATION / FINETUNE LAB (research only)
# Role: generate stress data, validate parser, prep (NOT run) LoRA finetune.
# DO NOT: deploy, serve live, promote rules. Outputs feed Account C audit only.
import os; os.makedirs("/kaggle/working/augment",exist_ok=True)
print("ACCOUNT A lab")

In [ ]:
# --- v40 engine (SYSTEM-UNDER-TEST, frozen copy) ---
# -*- coding: utf-8 -*-
"""v40.3 entity-grounded conjunctive Horn engine for the REAL Phase-1 distribution.
Propositional over a single entity: facts = literals, rules = (conj of literals)->literal.
Forward-chain; answer options by derivability. Certificate = premise indices. Abstain-safe."""
import re
STOP={'a','an','the','of','to','in','on','at','for','and','or','that','this','their','its','it','they','them',
      'is','are','was','were','be','been','has','have','had','then','if','no','not','with','as','by','from',
      'artifact','package','manuscript','sample','batch','item','device','record','file','student'}
def _stem(t):
    if re.search(r'(ss|us|is)$',t): return t
    if re.search(r'(ches|shes|xes|zes|ses)$',t): return t[:-2]
    if re.search(r'ies$',t): return t[:-3]+'y'
    if t.endswith('s'): t=t[:-1]
    return re.sub(r'(ing|ed)$','',t)
def atom_key(phrase):
    s=re.sub(r'(?<!^)(?=[A-Z])',' ',str(phrase)).lower()
    nums=re.findall(r'\d+', s)
    toks=[ _stem(w) for w in re.findall(r'[a-zA-Z]+', s) ]
    toks=[t for t in toks if t and t not in STOP and len(t)>2]
    keys=sorted(set(toks))+["N"+n for n in sorted(set(nums))]   # keep numeric literals
    return "".join(w.capitalize() for w in keys) if keys else None

# split a clause into (atom, neg). Handles "X has Y", "X is Y", "X has no Y", "X lacks Y", "cannot ...", "is not ..."
_LEAD=re.compile(r"^\s*(if|then|that|who|which|it|its|their|this)\b",re.I)
_VERB=re.compile(r"\b(cannot|can not|can|could|may|might|must|should|shall|will|would|is not|are not|was not|were not|isn't|aren't|is|are|was|were|has no|have no|had no|has|have|had|lacks?|without|requires?|needs?|contains?|completed?|enters?|gains?|receives?|provides?|shows?|states?|holds?|carries|monitors?|captures?|eligible|allowed|approved|assigned|be|been|being)\b",re.I)
ACTION_VERBS={'receives','receive','provides','provide','shows','show','states','state','monitors','monitor','captures','capture','enters','enter','requires','require','needs','need','gains','gain','completed','complete','contains','contain','reports','report','releases','release','passes','pass','improves','improve','supports','support','recommends','recommend','administered','administer','approved','approve'}
_NEGWORD=re.compile(r'\b(no|not|cannot|can not|lacks?|without|isn\'t|aren\'t|never|nor|incomplete|missing|lacking)\b',re.I)
def to_literal(clause):
    c=clause.strip().rstrip('.').strip()
    c=_LEAD.sub('',c).strip()
    neg=bool(re.search(r"\b(no|not|cannot|can not|never|lacks?|without|isn't|aren't|incomplete|missing|lacking|nor|un(?:able|verified|established))\b",c,re.I))
    m=_VERB.search(c)
    pred=c[m.end():].strip() if m else c
    verb=(m.group(1).lower() if m else '')
    if m and verb in ACTION_VERBS:
        pred = verb + ' ' + pred
    # peel any leftover leading modal/aux/passive markers and articles
    for _ in range(4):
        pred=re.sub(r"^\s*(be|been|being|to|a|an|the|no|not|its|their)\b","",pred,flags=re.I).strip()
    a=atom_key(pred)
    return (a,neg) if a else None

def parse_premise(p):
    s=str(p).strip()
    m=re.search(r'^\s*if\b(.+?),?\s*\bthen\b(.+)$',s,re.I)
    if m:
        ante=re.split(r'\band\b',m.group(1),flags=re.I)
        lits=[to_literal(x) for x in ante]; lits=[l for l in lits if l]
        con=to_literal(m.group(2))
        if con and lits: return ('rule',lits,con)
        return None
    # Universal relative rule: Every/All <role> <condition> <verb> <consequent>
    # Examples: Every volunteer assigned to shift receives badge; All satellites with cameras can capture images.
    m2=re.search(r'^\s*(every|all)\s+[a-zA-Z]+s?\s+(.+?)\s+\b(can|may|must|should|will|would|receives?|gets?|gains?|provides?|captures?|monitors?|requires?|needs?|is|are)\b\s+(.+)$',s,re.I)
    if m2:
        cond=m2.group(2).strip()
        cons=(m2.group(3)+" "+m2.group(4)).strip()
        litc=to_literal(cond); litd=to_literal(cons)
        if litc and litd: return ('rule',[litc],litd)
    if re.search(r'^\s*(no premise|it (is|cannot)|unknown|there is no information)',s,re.I): return None
    lit=to_literal(s)
    return ('fact',lit) if lit else None

def solve_entity(premises):
    facts={}  # atom -> (bool_value, premise_idx)
    rules=[]
    for i,p in enumerate(premises):
        pp=parse_premise(p)
        if not pp: continue
        if pp[0]=='fact':
            a,neg=pp[1]; facts.setdefault(a,(not neg,[i]))
        else:
            rules.append((i,pp[1],pp[2]))
    changed=True
    while changed:
        changed=False
        for i,lits,con in rules:
            ca,cneg=con
            ok=True; path=[i]
            for a,neg in lits:
                if a in facts and facts[a][0]==(not neg): path+=facts[a][1]
                else: ok=False; break
            if ok and ca not in facts:
                facts[ca]=((not cneg),sorted(set(path))); changed=True
    return facts

_META_RE=__import__("re").compile(r"\b(not (?:yet )?(?:established|confirmed|verified|approved|cleared|determined)|cannot be (?:established|confirmed)|unsupported|is not established|no premise|undetermined|not (?:available|present))\b",__import__("re").I)
def decompose_option(opt):
    import re
    t=re.sub(r'^\s*[A-Da-d][.):]\s*','',str(opt)).strip()
    t=re.split(r'\bbecause\b', t, maxsplit=1, flags=re.I)[0].strip()  # drop causal justification
    parts=re.split(r',\s*but\s+|\s+but\s+|;\s+|\s+while\s+|\s+whereas\s+|\s+and\s+',t,flags=re.I)
    claims=[]
    for p in parts:
        p=p.strip()
        if not p: continue
        is_meta=bool(_META_RE.search(p))
        lit=to_literal(p)
        claims.append((lit,is_meta,p))
    return claims

def answer_mc(premises, options):
    facts=solve_entity(premises)
    res={}
    for lab,opt in zip("ABCD",options):
        claims=decompose_option(opt)
        if not claims: res[lab]=('UNSUP',[]); continue
        status='PROVEN'; path=[]
        for lit,is_meta,txt in claims:
            if lit is None: status='UNSUP'; break
            a,neg=lit; have = a in facts
            val = facts[a][0] if have else None
            if is_meta:
                # meta "not established": correct only if NOT positively proven
                if have and val==True: status='DISPROVEN'; break
            else:
                if have and val==(not neg): path+=facts[a][1]
                elif have and val==(neg): status='DISPROVEN'; break
                else: status='UNSUP'; break
        res[lab]=(status, sorted(set(path)))
    proven=[l for l in res if res[l][0]=='PROVEN']
    if len(proven)==1: return proven[0],res[proven[0]][1],'entity_unique_proof',res
    return None,[],('multiple' if proven else 'none'),res


# ================= Phase-1 REPLAY HARNESS =================
def _opt_texts(rp):
    import re
    f=[o[1].strip().replace("\n"," ") for o in re.findall(r"(?:^|\n)\s*([A-D])[.)]\s*(.+?)(?=\n\s*[A-D][.)]|\Z)",rp.get("query",""),flags=re.S)]
    return f if len(f)>=2 else (rp.get("options") or [])
def replay_phase1(path):
    import json,re
    d=json.load(open(path)); t1=[l for l in d["logs"] if l.get("type")=="type1"]
    fired=aok=pok=0; fixed=[]; wrong=[]; abst=0
    for l in t1:
        rp=l["request_payload"]; exp=l.get("expected") or {}; ea=str(exp.get("answer","")).strip().upper()
        opts=_opt_texts(rp)
        if not opts: continue
        a,pu,why,res=answer_mc(rp.get("premises",[]) or [], opts)
        if a is None: abst+=1; continue
        fired+=1; ok=(a==ea); aok+=ok; pok+=(sorted(pu)==sorted(exp.get("premises_used") or []))
        if ok and l.get("status")!="correct": fixed.append(l["query_id"])
        if not ok: wrong.append({"query_id":l["query_id"],"exp":ea,"got":a})
    rep={"n":len(t1),"fired":fired,"answer_correct":aok,"premises_used_correct":pok,
         "wrong":wrong,"fixed_old_wrong":fixed,"abstained":abst,
         "precision_when_fired":round(aok/max(fired,1),3),"coverage":round(fired/max(len(t1),1),3),
         "gate":"ABSTAIN_SAFE" if not wrong else "HAS_WRONG_FIX_BEFORE_APPLY"}
    return rep
def _autofind():
    import glob,os,sys
    if len(sys.argv)>1 and os.path.exists(sys.argv[1]): return sys.argv[1]
    for c in ["exact_eval_round1_Astatine.json","/kaggle/input/**/exact_eval_round1_Astatine.json",
              "/kaggle/working/exact_eval_round1_Astatine.json","./exact_eval_round1_Astatine.json"]:
        h=sorted(glob.glob(c,recursive=True)) if any(x in c for x in "*?[") else ([c] if os.path.exists(c) else [])
        if h: return h[0]
    return None
print('v40 engine loaded (SUT)')

In [ ]:
# --- hardened anti-circular augmenter ---
# -*- coding: utf-8 -*-
"""type1_aug_paraphrase_skeleton.py (hardened) — anti-circular Type1 parser stress augmenter.
Each sample carries an ABSTRACT skeleton (atoms P1/P2/.., rules, option->abstract-claim).
Validator = abstract reference solver (independent of v40): guarantees correct-unique,
distractors-not-provable, canary->>=2 proven. v40 is SYSTEM-UNDER-TEST on paraphrased surface.
Use for parser stress-test + shadow-eval ONLY. Not for LoRA training; not sole promotion evidence.
"""
import json,random,re,sys,os
random.seed(11)

TOPICS=[
 {"ent":"MedKit-7","P1":["launch clearance","clearance for launch"],"P2":["a blocked-route report","a route-blocking flag"],"P3":["eligible to use the aerial corridor","permitted on the aerial corridor"]},
 {"ent":"The River Codex","P1":["a 600 dpi scan","a high-resolution scan"],"P2":["a privacy flag","an unresolved privacy issue"],"P3":["eligible for the public portal","portal-eligible"]},
 {"ent":"Satellite Vega","P1":["calibrated thermal sensors","thermal sensors that are calibrated"],"P2":["a radar fault","a sensor fault"],"P3":["able to support disaster mapping","disaster-mapping capable"]},
 {"ent":"Batch Nova","P1":["an intact seal","a seal that is intact"],"P2":["a temperature breach","a cold-chain breach"],"P3":["release-ready","ready for release"]},
]
def _np(ph): return re.sub(r'^(a|an)\s+','',ph)
def say_pos(e,ph): return random.choice([f"{e} has {ph}.",f"{_np(ph).capitalize()} is present for {e}.",f"{e} carries {ph}."])
def say_neg(e,ph): p=_np(ph); return random.choice([f"{e} has no {p}.",f"No {p} is recorded for {e}.",f"{e} lacks {p}."])
def say_state(e,ph): return random.choice([f"{e} is {ph}.",f"{e} becomes {ph}.",f"{_np(ph).capitalize()} applies to {e}."])
def rule_para(a,b,c):  # surface variety of: has A and no B -> C  (STRESS the parser)
    bp=_np(b)
    return random.choice([
      f"If an item has {a} and no {bp}, then it is {c}.",
      f"When an item has {a} but no {bp}, it is {c}.",
      f"Any item with {a} and without {bp} is {c}.",
      f"An item that has {a} and lacks {bp} is {c}."])

# ---- abstract reference solver (independent of v40) ----
def abstract_solve(spec):
    facts=dict(spec["facts"])  # atom->bool
    for _ in range(8):
        ch=False
        for (ante,con) in spec["rules"]:
            if all(facts.get(a)==v for a,v in ante) and con[0] not in facts:
                facts[con[0]]=con[1]; ch=True
        if not ch: break
    return facts
def claim_status(claim, facts):
    a,v=claim
    if a in facts: return "PROVEN" if facts[a]==v else "DISPROVEN"
    return "UNSUP"

def gen_conjunctive(canary=False):
    t=random.choice(TOPICS); e=t["ent"]; a=random.choice(t["P1"]); b=random.choice(t["P2"]); c=random.choice(t["P3"])
    prem=[rule_para(a,b,c), say_pos(e,a), say_neg(e,b)]
    spec={"facts":{"P1":True,"P2":False},"rules":[([("P1",True),("P2",False)],("P3",True))],
          "options":{}}
    # options + abstract claims
    optmap={"A":(f"{e} has {_np(b)}",("P2",True)),    # FALSE (P2 is false)
            "B":(f"{e} has an audit waiver",("WAIV",True)),  # unsupported
            "C":(f"{e} is {c}",("P3",True)),           # correct
            "D":(f"{e} is not {c}",("P3",False))}      # contradiction
    if canary:
        # make B genuinely also-proven: add fact P1 claim as option B (P1 true) + add a second true fact
        optmap["B"]=(f"{e} has {a}",("P1",True))
    opts=[f"{lab}. {txt}" for lab,(txt,_) in optmap.items()]
    spec["options"]={lab:cl for lab,(_,cl) in optmap.items()}
    ans=None if canary else "C"; pu=[] if canary else [0,1,2]
    return {"logic_family":"entity_conjunctive_mc","surface_family":f"para_{e.split()[-1].lower()}",
            "entity":e,"premises":prem,"query":"Based on the premises, which statement is correct?\n"+"\n".join(opts),
            "options":opts,"answer":ans,"premises_used":pu,"canary":canary,"abstract":spec}

def gen_meta_unknown():
    t=random.choice(TOPICS); e=t["ent"]; a=random.choice(t["P1"]); x=random.choice(["budget approval","export license"])
    prem=[say_pos(e,a), f"No premise states whether {e} has {x}."]
    # YNU-style but presented as MC with Uncertain option
    optmap={"A":(f"{e} has {x}",("X",True)),"B":(f"{e} has no {x}",("X",False)),
            "C":(f"It cannot be determined whether {e} has {x}",("META",True)),"D":(f"{e} has {a}",("P1",True))}
    spec={"facts":{"P1":True},"rules":[],"options":{lab:cl for lab,(_,cl) in optmap.items()}}
    opts=[f"{lab}. {txt}" for lab,(txt,_) in optmap.items()]
    # correct = C (meta) since X undetermined; but D(P1) is also true -> ambiguous! so gold=D? keep simple: drop D-as-true
    optmap["D"]=(f"{e} lacks {a}",("P1",False)); opts[3]=f"D. {e} lacks {_np(a)}"
    spec["options"]["D"]=("P1",False)
    return {"logic_family":"meta_unknown","surface_family":"meta","entity":e,
            "premises":prem,"query":"Which statement is correct?\n"+"\n".join(opts),
            "options":opts,"answer":"C","premises_used":[1],"canary":False,"abstract":spec,"meta_option":"C"}

GENS=[("conjunctive",lambda:gen_conjunctive(False)),("canary",lambda:gen_conjunctive(True)),("meta",gen_meta_unknown)]

def make(n,split):
    out=[]
    for i in range(n):
        nm,gn=random.choice(GENS); s=gn(); s["sample_id"]=f"{split}_{i:04d}"; s["split"]=split; out.append(s)
    return out

# ---- HARDENED validator: abstract reference solver, reject ill-formed ----
def validate(rows):
    rep={"total":len(rows),"rejected":0,"reasons":{}}; keep=[]
    for s in rows:
        facts=abstract_solve(s["abstract"]); opts=s["abstract"]["options"]
        # meta option: treated as PROVEN iff no other DIRECT option proven
        direct_proven=[l for l,cl in opts.items() if cl[0]!="META" and claim_status(cl,facts)=="PROVEN"]
        proven=set(direct_proven)
        if s.get("meta_option") and not direct_proven: proven.add(s["meta_option"])
        bad=None
        if s["canary"]:
            if len(proven)<2: bad="canary_not_multi_proven"
        else:
            if len(proven)!=1: bad="not_unique_proven"
            elif list(proven)[0]!=s["answer"]: bad="proven_mismatch_gold"
            elif not s["premises_used"]: bad="empty_premises_used"
        if bad: rep["rejected"]+=1; rep["reasons"][bad]=rep["reasons"].get(bad,0)+1
        else: keep.append(s)
    return keep,rep

# ---- parser stress: v40 on surface vs abstract-validated gold ----
def stress(rows, engine):
    src=open(engine).read().split("if __name__")[0]; g={}; exec(src,g); am=g["answer_mc"]
    fired=ok=puok=wrong=can_ok=can_n=0; groups={}
    for s in rows:
        a,pu,why,res=am(s["premises"],s["options"])
        if s["canary"]:
            can_n+=1; can_ok+=(a is None); continue
        if a is None: continue
        fired+=1; c=(str(a).upper()==str(s["answer"]).upper()); ok+=c; puok+=(sorted(pu)==sorted(s["premises_used"])); wrong+=(not c)
        groups.setdefault(s["surface_family"],set()).add(a)
    cons=[len(v)==1 for v in groups.values() if len(v)>1]
    return {"n":len(rows),"fired":fired,"correct_when_fired":ok,"premises_used_correct":puok,
            "wrong_real_parser":wrong,"canary_n":can_n,"canary_abstained":can_ok,
            "paraphrase_consistency":f"{sum(cons)}/{len(cons)}" if cons else "n/a"}

In [ ]:
# --- RUN augmenter: dev/holdout + validator + parser stress (v40 as SUT) ---
import json
DEV=int(os.environ.get("AUG_DEV","80")); HOLD=int(os.environ.get("AUG_HOLD","200"))
dev,vd=validate(make(DEV,"dev")); hold,vh=validate(make(HOLD,"holdout"))
OUT="/kaggle/working/augment"
json.dump(dev,open(f"{OUT}/type1_aug_dev.json","w"),indent=1)
json.dump(hold,open(f"{OUT}/type1_aug_holdout.json","w"),indent=1)
json.dump({"dev":vd,"holdout":vh},open(f"{OUT}/type1_aug_validator_report.json","w"),indent=1)
# stress test: v40 (answer_mc defined in engine cell) on surface
def _stress(rows):
    fired=ok=puok=wrong=can_ok=can_n=0
    for s in rows:
        a,pu,why,res=answer_mc(s["premises"],s["options"])
        if s["canary"]: can_n+=1; can_ok+=(a is None); continue
        if a is None: continue
        fired+=1; c=(str(a).upper()==str(s["answer"]).upper()); ok+=c; puok+=(sorted(pu)==sorted(s["premises_used"])); wrong+=(not c)
    return {"n":len(rows),"fired":fired,"correct_when_fired":ok,"premises_used_correct":puok,"wrong_real_parser":wrong,"canary_n":can_n,"canary_abstained":can_ok}
st={"dev":_stress(dev),"holdout":_stress(hold)}
json.dump(st,open(f"{OUT}/type1_aug_parser_stress_report.json","w"),indent=1)
print(json.dumps(st,indent=1))

In [ ]:
# --- LoRA finetune SCAFFOLD (NOT RUN) ---
# Gate before any retrain: parser/certificate stable on aug_holdout AND fallback still default-to-A heavy.
# Train ONLY a fallback answer-selector; symbolic layer still owns premises_used. Keep Final Answer format.
RUN_LORA_FINETUNE=False
if RUN_LORA_FINETUNE:
    raise SystemExit("Finetune intentionally gated off. Enable only after augmentation+parser validated (Account C).")
print("LoRA finetune gated OFF (by design).")